# Phase 7 — Incremental Processing

**Objectifs:**
- [x] Détection des nouveaux fichiers
- [x] Contrôle des doublons
- [x] MERGE
- [x] Historique d'ingestion

**Livrable:** Incremental Pipeline

**Problème:** chaque mois, TLC publie un nouveau fichier. Recharger **tout** l'historique à chaque fois coûte de plus en plus cher.

**Solution incrémentale:** ne traiter **que les nouveaux fichiers**, et les intégrer en Silver avec un **MERGE** (upsert) → le pipeline est **idempotent**: le relancer 2 fois donne le même résultat.

```
landing/ ──► détecter fichiers ∉ ingestion_log ──► append Bronze (+ log) ──► bronze_to_silver(batch) ──► MERGE Silver
```

## 0. Setup

In [1]:
from lakehouse_config import *
from lakehouse_transforms import *
spark = get_spark()
setup_schemas(spark)

from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import uuid

In [2]:
def table_counts():
    return {t: spark.table(t).count() for t in [T_BRONZE_TRIPS, T_SILVER_TRIPS, T_SILVER_REJECTED]}

counts_before = table_counts()
counts_before

{'workspace.bronze.yellow_trips': 2964624,
 'workspace.silver.yellow_trips': 2869912,
 'workspace.silver.yellow_trips_rejected': 94712}

## 1. Détection des nouveaux fichiers
### 1.1 Arrivée d'un nouveau fichier (simulation: février 2024)

In [3]:
NEW_MONTH = "2024-02"
upload_to_volume(TRIPS_URL.format(month=NEW_MONTH), f"yellow_tripdata_{NEW_MONTH}.parquet")

Telechargement: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet


Upload: /Volumes/workspace/raw/landing/yellow_tripdata_2024-02.parquet


'/Volumes/workspace/raw/landing/yellow_tripdata_2024-02.parquet'

### 1.2 Fichiers présents vs fichiers déjà ingérés

In [4]:
def detect_new_files():
    w = get_workspace_client()
    landing = {f.name for f in w.files.list_directory_contents(RAW_VOLUME)
               if f.name.startswith("yellow_tripdata_") and f.name.endswith(".parquet")}
    already = {r.file_name for r in spark.table(T_INGESTION_LOG).filter("status = 'SUCCESS'")
                                         .select("file_name").distinct().collect()}
    return sorted(landing - already), sorted(landing), sorted(already)


new_files, landing_files, ingested_files = detect_new_files()
print("Dans landing    :", landing_files)
print("Deja ingeres    :", ingested_files)
print("NOUVEAUX        :", new_files)

Dans landing    : ['yellow_tripdata_2024-01.parquet', 'yellow_tripdata_2024-02.parquet']
Deja ingeres    : ['yellow_tripdata_2024-01.parquet']
NOUVEAUX        : ['yellow_tripdata_2024-02.parquet']


## 2. Contrôle des doublons

Deux niveaux de protection:
1. **Niveau fichier:** un fichier déjà présent dans `ingestion_log` (statut `SUCCESS`) n'est **jamais** relu.
2. **Niveau ligne:** le **MERGE sur `trip_id`** n'insère pas une course déjà présente en Silver (même si elle arrive dans un autre fichier).

### 2.1 Ingestion Bronze des nouveaux fichiers (append + log)

In [5]:
def log_ingestion(file_name, rows, status, batch_id, error=None):
    spark.createDataFrame(
        [(file_name, T_BRONZE_TRIPS, rows, status, "incremental", batch_id)],
        "file_name string, target_table string, row_count long, status string, load_type string, batch_id string",
    ).withColumn("ingested_at", F.current_timestamp()) \
     .write.format("delta").mode("append").saveAsTable(T_INGESTION_LOG)
    if error:
        print(f"  ERREUR {file_name}: {error}")


def ingest_new_files(files):
    batch_ids = []
    for f in files:
        batch_id = str(uuid.uuid4())
        try:
            df = add_ingestion_metadata(spark.read.parquet(f"{RAW_VOLUME}/{f}"), batch_id)
            # mergeSchema: tolere une nouvelle colonne ajoutee par TLC dans un fichier recent
            df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(T_BRONZE_TRIPS)
            rows = spark.table(T_BRONZE_TRIPS).filter(F.col("_batch_id") == batch_id).count()
            log_ingestion(f, rows, "SUCCESS", batch_id)
            batch_ids.append(batch_id)
            print(f"  {f}: {rows:,} lignes ajoutees en Bronze (batch {batch_id[:8]})")
        except Exception as e:
            log_ingestion(f, 0, "FAILED", batch_id, error=str(e).splitlines()[0][:200])
    return batch_ids


new_batch_ids = ingest_new_files(new_files)

  yellow_tripdata_2024-02.parquet: 3,007,526 lignes ajoutees en Bronze (batch 87e0e5bb)


## 3. MERGE

`MERGE INTO` = **upsert** atomique:
- `WHEN MATCHED` → la course existe déjà (même `trip_id`) → on la met à jour.
- `WHEN NOT MATCHED` → nouvelle course → on l'insère.

On ne transforme **que le(s) nouveau(x) batch(s)** Bronze, pas tout l'historique.

In [6]:
def merge_batches_into_silver(batch_ids):
    if not batch_ids:
        print("Rien a merger.")
        return None
    batch = spark.table(T_BRONZE_TRIPS).filter(F.col("_batch_id").isin(batch_ids))
    valid, rejected = bronze_to_silver(batch)

    valid.createOrReplaceTempView("silver_updates")
    merge_metrics = spark.sql(f"""
        MERGE INTO {T_SILVER_TRIPS} AS t
        USING silver_updates AS s
        ON t.trip_id = s.trip_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    add_trip_id(rejected).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(T_SILVER_REJECTED)
    return merge_metrics


metrics = merge_batches_into_silver(new_batch_ids)
if metrics is not None:
    metrics.show()

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|          2901708|               0|               0|          2901708|
+-----------------+----------------+----------------+-----------------+



In [7]:
counts_after = table_counts()
pd.DataFrame({"avant": counts_before, "apres": counts_after}).assign(delta=lambda d: d.apres - d.avant)

,avant,apres,delta
workspace.bronze.yellow_trips,2964624,5972150,3007526
workspace.silver.yellow_trips,2869912,5771620,2901708
workspace.silver.yellow_trips_rejected,94712,200529,105817


### 3.1 Idempotence — on relance le pipeline: rien ne doit changer

In [8]:
new_files_2, _, _ = detect_new_files()
print("Nouveaux fichiers au 2eme passage:", new_files_2)
merge_batches_into_silver(ingest_new_files(new_files_2))

assert table_counts() == counts_after, "Le pipeline n'est pas idempotent !"
print("Idempotent: les tables n'ont pas change.", table_counts())

Nouveaux fichiers au 2eme passage: []
Rien a merger.


Idempotent: les tables n'ont pas change. {'workspace.bronze.yellow_trips': 5972150, 'workspace.silver.yellow_trips': 5771620, 'workspace.silver.yellow_trips_rejected': 200529}


In [9]:
# Controle: aucun doublon en Silver apres le MERGE
spark.sql(f"SELECT COUNT(*) AS lignes, COUNT(DISTINCT trip_id) AS trip_id_uniques FROM {T_SILVER_TRIPS}").show()
spark.sql(f"SELECT date_format(pickup_date, 'yyyy-MM') AS mois, COUNT(*) AS courses FROM {T_SILVER_TRIPS} GROUP BY 1 ORDER BY 1").show()

+-------+---------------+
| lignes|trip_id_uniques|
+-------+---------------+
|5771620|        5771620|
+-------+---------------+



+-------+-------+
|   mois|courses|
+-------+-------+
|2024-01|2869912|
|2024-02|2901708|
+-------+-------+



## 4. Historique d'ingestion

In [10]:
spark.table(T_INGESTION_LOG).orderBy("ingested_at").select(
    "file_name", "load_type", "status", "row_count", "batch_id", "ingested_at").show(truncate=False)

+-------------------------------+-----------+-------+---------+------------------------------------+--------------------------+
|file_name                      |load_type  |status |row_count|batch_id                            |ingested_at               |
+-------------------------------+-----------+-------+---------+------------------------------------+--------------------------+
|yellow_tripdata_2024-01.parquet|full_load  |SUCCESS|2964624  |78a6d09b-28bc-4911-8d7e-0d300e0bb9cf|2026-11-08 19:02:50.517122|
|yellow_tripdata_2024-02.parquet|incremental|SUCCESS|3007526  |87e0e5bb-dd4d-4889-a241-0d0f55036b5b|2026-11-08 19:08:27.508162|
+-------------------------------+-----------+-------+---------+------------------------------------+--------------------------+



In [11]:
# Historique Delta: chaque append / MERGE est une version
spark.sql(f"DESCRIBE HISTORY {T_SILVER_TRIPS}").select(
    "version", "timestamp", "operation", "operationMetrics.numTargetRowsInserted",
    "operationMetrics.numTargetRowsUpdated", "operationMetrics.numOutputRows").show(10, truncate=False)

+-------+-------------------+---------------------------------+---------------------+--------------------+-------------+
|version|timestamp          |operation                        |numTargetRowsInserted|numTargetRowsUpdated|numOutputRows|
+-------+-------------------+---------------------------------+---------------------+--------------------+-------------+
|2      |2026-11-08 19:08:46|MERGE                            |2901708              |0                   |2901708      |
|1      |2026-11-08 19:04:23|SET TBLPROPERTIES                |NULL                 |NULL                |NULL         |
|0      |2026-11-08 19:04:13|CREATE OR REPLACE TABLE AS SELECT|NULL                 |NULL                |2869912      |
+-------+-------------------+---------------------------------+---------------------+--------------------+-------------+



## Conclusion — Incremental Pipeline

- **Détection**: `fichiers landing − fichiers SUCCESS dans ingestion_log` = fichiers à traiter.
- **Bronze**: append avec `_batch_id` + ligne dans `ingestion_log` (statut `SUCCESS` / `FAILED`).
- **Silver**: seul le nouveau batch est transformé, puis intégré par **MERGE sur `trip_id`** → aucun doublon.
- **Idempotent**: relancer le pipeline ne change rien (vérifié par `assert`).
- **Historique** complet: `ops.ingestion_log` (niveau fichier) + `DESCRIBE HISTORY` (niveau table).
- ⚠️ Après un incrément, il faut **recalculer Gold** (Phase 5) puis relancer la **Data Quality** (Phase 6) → c'est ce que fait le **Workflow** de la Phase 9.